In [1]:
import torch
from deel import torchlip
from VGG_Arthur import load_model, HKRMultiLossLSE
from training_VGG_arthur import load_cifar10, main
import schedulefree as sf
from torchvision import datasets
from torchvision.transforms import v2
from torch.utils.data import DataLoader
import yaml
import torch.nn as nn
import torch.nn.functional as F
import time
from torchinfo import summary

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# Load configuration file
with open('config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

In [4]:
# Load dataset
train_loader, test_loader = load_cifar10(cfg)

/home/aws_install/miniconda3/envs/k3torchenv/lib/python3.10/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [5]:
model_b= load_model().vanilla_export().to(device)
model_b.load_state_dict(torch.load('/home/aws_install/robustess_project/lip_notebooks/notebooks_creation_models/Vgg_lip_multisteplr_van.pt', weights_only=True))
# model_b.eval()

/home/aws_install/miniconda3/envs/k3torchenv/lib/python3.10/site-packages/deel/torchlip/modules/module.py:159: UserWarning: Sequential model contains a layer which is not a Lipschitz layer: LipBlock(
  (conv): ParametrizedSpectralConv2d(
    3, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect
    (parametrizations): ModuleDict(
      (weight): ParametrizationList(
        (0): _SpectralNorm()
        (1): _BjorckNorm()
        (2): _LConvNorm()
      )
    )
  )
  (norm): BatchCentering()
  (activation): GroupSort2()
  (scalar): MultiplyByScalar()
)
  warnings.warn(
/home/aws_install/miniconda3/envs/k3torchenv/lib/python3.10/site-packages/deel/torchlip/modules/module.py:159: UserWarning: Sequential model contains a layer which is not a Lipschitz layer: LipBlock(
  (conv): ParametrizedSpectralConv2d(
    96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect
    (parametrizations): ModuleDict(
      (weight): Pa

<All keys matched successfully>

In [6]:
model_c= load_model().to(device)
model_c.load_state_dict(torch.load('/home/aws_install/robustess_project/lip_notebooks/notebooks_creation_models/Vgg_lip_multisteplr.pt', weights_only=True))
# model_c.eval()

<All keys matched successfully>

In [ ]:
model_b.eval()
model_c.eval()
for img, lb in test_loader:
    break
prediction = model_b(torch.ones_like(img.to(device))*1)
print(prediction)
prediction = model_c(torch.ones_like(img.to(device))*1)
print(prediction)

tensor([[-0.0008, -0.0007, -0.0013,  ..., -0.0016, -0.0008, -0.0017],
        [-0.0008, -0.0007, -0.0013,  ..., -0.0016, -0.0008, -0.0017],
        [-0.0008, -0.0007, -0.0013,  ..., -0.0016, -0.0008, -0.0017],
        ...,
        [-0.0008, -0.0007, -0.0013,  ..., -0.0016, -0.0008, -0.0017],
        [-0.0008, -0.0007, -0.0013,  ..., -0.0016, -0.0008, -0.0017],
        [-0.0008, -0.0007, -0.0013,  ..., -0.0016, -0.0008, -0.0017]],
       device='cuda:0', grad_fn=<MulBackward0>)
tensor([[-0.0008, -0.0007, -0.0013,  ..., -0.0016, -0.0008, -0.0017],
        [-0.0008, -0.0007, -0.0013,  ..., -0.0016, -0.0008, -0.0017],
        [-0.0008, -0.0007, -0.0013,  ..., -0.0016, -0.0008, -0.0017],
        ...,
        [-0.0008, -0.0007, -0.0013,  ..., -0.0016, -0.0008, -0.0017],
        [-0.0008, -0.0007, -0.0013,  ..., -0.0016, -0.0008, -0.0017],
        [-0.0008, -0.0007, -0.0013,  ..., -0.0016, -0.0008, -0.0017]],
       device='cuda:0', grad_fn=<MulBackward0>)


In [20]:
img.shape

torch.Size([128, 3, 32, 32])

In [8]:
x,y = next(iter(test_loader))

In [10]:
model_b(x.to(device)).argmax(dim=1), y

(tensor([3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6, 7, 0, 4, 9,
         4, 2, 4, 0, 9, 6, 6, 5, 4, 5, 9, 3, 4, 9, 9, 5, 4, 6, 5, 6, 0, 9, 3, 9,
         7, 6, 9, 8, 0, 3, 8, 8, 7, 7, 7, 6, 7, 5, 6, 3, 6, 2, 1, 2, 3, 7, 2, 6,
         8, 8, 0, 2, 9, 3, 5, 8, 8, 1, 1, 7, 2, 7, 2, 7, 8, 9, 0, 3, 8, 6, 4, 6,
         6, 0, 0, 7, 4, 5, 6, 3, 1, 1, 3, 6, 8, 7, 4, 0, 2, 2, 1, 3, 0, 4, 2, 7,
         8, 3, 1, 2, 8, 0, 8, 3], device='cuda:0'),
 tensor([3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6, 7, 0, 4, 9,
         5, 2, 4, 0, 9, 6, 6, 5, 4, 5, 9, 2, 4, 1, 9, 5, 4, 6, 5, 6, 0, 9, 3, 9,
         7, 6, 9, 8, 0, 3, 8, 8, 7, 7, 4, 6, 7, 3, 6, 3, 6, 2, 1, 2, 3, 7, 2, 6,
         8, 8, 0, 2, 9, 3, 3, 8, 8, 1, 1, 7, 2, 5, 2, 7, 8, 9, 0, 3, 8, 6, 4, 6,
         6, 0, 0, 7, 4, 5, 6, 3, 1, 1, 3, 6, 8, 7, 4, 0, 6, 2, 1, 3, 0, 4, 2, 7,
         8, 3, 1, 2, 8, 0, 8, 3]))

In [13]:
# model.train() # https://github.com/facebookresearch/schedule_free
# optimizer.eval()
model_c.eval()
            
# model_b.eval()
    ############################ Test loop #############################
total_correct,total_loss = 0, 0
    #all_logits, all_labels = [], []
for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    labels_onehot = nn.functional.one_hot(labels, 10)

    with torch.no_grad():
        logits = model_c(images)

        # loss = criterion(logits, labels_onehot)

            #all_logits.append(logits)
            #all_labels.append(labels)

        total_correct += (logits.argmax(dim=1) == labels).sum().item()
            #total_adv_correct += (adv_logits.argmax(dim=1) == labels).sum().item()
        # total_loss += loss.item()

# model.eval()
        #all_logits = torch.cat(all_logits)
        #all_labels = torch.cat(all_labels)

test_accuracy = 100 * total_correct / len(test_loader.dataset)
    #test_aa_accuracy = 100 * total_adv_correct / len(test_loader.dataset)
# test_loss = total_loss / len(test_loader.dataset)
           
            
# end_time = time.time()
# time_elapsed = end_time - start_time
            
     

    # Print metrics
print(test_accuracy)

88.52


In [15]:
import sys
sys.path.append('..')
from radius_evaluation_tools_torch import *
from data_processing_torch import *

In [16]:
print("Generating Sample :")
images, labels = select_data_for_radius_evaluation(test_loader, test_loader.dataset, model_b, schedulefree=True)
images = images.to(device)
labels = labels.to(device)

total_points = images.shape[0]


print("Generating Certificates :")
lip_radius = compute_certificate(images, model_b)

Generating Sample :
Generating Certificates :


In [18]:
lip_radius[0]

tensor(0.0020, device='cuda:0', grad_fn=<SelectBackward0>)

In [17]:
images[0]

tensor([[[0.6078, 0.6549, 0.6902,  ..., 0.7882, 0.7922, 0.7529],
         [0.6000, 0.6392, 0.6706,  ..., 0.7922, 0.7961, 0.7412],
         [0.6078, 0.6275, 0.6588,  ..., 0.8078, 0.8000, 0.7412],
         ...,
         [0.3490, 0.2235, 0.2392,  ..., 0.3490, 0.2314, 0.2627],
         [0.3490, 0.2353, 0.2471,  ..., 0.2235, 0.2392, 0.2941],
         [0.3608, 0.2353, 0.2392,  ..., 0.2353, 0.2510, 0.2863]],

        [[0.6118, 0.6902, 0.7020,  ..., 0.7686, 0.7922, 0.7176],
         [0.6078, 0.7020, 0.7216,  ..., 0.8157, 0.8431, 0.7451],
         [0.6039, 0.6980, 0.7255,  ..., 0.8157, 0.8431, 0.7490],
         ...,
         [0.3098, 0.2078, 0.2392,  ..., 0.4667, 0.3098, 0.2902],
         [0.3216, 0.2275, 0.2588,  ..., 0.2588, 0.2510, 0.2706],
         [0.3059, 0.2039, 0.2275,  ..., 0.2471, 0.2549, 0.2667]],

        [[0.5843, 0.7333, 0.7569,  ..., 0.8196, 0.8314, 0.6706],
         [0.6157, 0.8000, 0.8431,  ..., 0.8706, 0.8902, 0.7176],
         [0.6000, 0.7882, 0.8353,  ..., 0.8431, 0.8784, 0.